# 10강 실습 과제

## 나만의 대시보드 만들고 배포하기

**제출 방법**: 완성된 앱을 Streamlit Cloud에 배포한 후, 배포 URL을 eTL 과제함에 제출

**제출 기한**: 다음 수업 전날 자정까지

---

### 과제 목표

10강에서 배운 다양한 기능들을 활용하여 **주제를 직접 정하고** 나만의 대시보드를 만든다.  
9강 과제보다 자유도가 높다. 아래 조건만 충족하면 주제와 내용은 완전히 자유다.

### 필수 조건 (모두 포함해야 함)

| 조건 | 사용해야 할 기능 |
|---|---|
| 텍스트 | `st.title()`, `st.header()` 또는 `st.markdown()` 중 하나 이상 |
| 데이터 | `st.dataframe()`, `st.table()`, `st.metric()` 중 하나 이상 |
| 차트 | `st.bar_chart()`, `st.line_chart()`, `st.area_chart()` 중 하나 이상 |
| 위젯 | `st.slider()`, `st.selectbox()`, `st.radio()` 등 위젯 하나 이상 |
| 위젯-화면 연동 | 위젯 값이 바뀌면 화면 내용도 바뀌어야 함 |

### 과제 순서 한눈에 보기

```
STEP 1.  주제 정하기
STEP 2.  대시보드 만들기
STEP 3.  로컬에서 실행 확인
STEP 4.  requirements.txt 생성
STEP 5.  GitHub 업로드
STEP 6.  Streamlit Cloud 배포 후 URL 제출
```

---

## STEP 1. 주제 정하기

대시보드의 주제를 먼저 정한다. 아래 예시를 참고하되, 본인이 관심 있는 주제면 무엇이든 좋다.

| 주제 예시 | 어떤 데이터를 쓸까 |
|---|---|
| 이번 학기 성적 관리 | 과목별 점수, 학점 |
| 나의 운동 기록 | 날짜별 운동 시간, 종류 |
| 용돈 지출 분석 | 카테고리별 지출 금액 |
| 좋아하는 영화/책 목록 | 장르, 평점, 연도 |
| 관심 있는 주식/코인 | 가격, 등락률 |
| 날씨 데이터 시각화 | 월별 기온, 강수량 |

> 데이터는 직접 만들어도 됩니다 (딕셔너리나 리스트로 하드코딩).  
> 실제 데이터를 CSV로 만들어서 써도 됩니다.

**내 주제**

- 주제: 오늘 뭐 볼까? — 영화 취향 대시보드
- 사용할 데이터: 영화 100편의 제목, 장르 8종, 과제용 예시 평점(2.0~9.8점)을 담은 `movies.csv`
- 만들려는 대시보드 설명 (1~2줄): 사이드바에서 장르, 최소 평점, 표시할 영화 수를 선택하면 영화 목록과 요약이 바뀐다. 라디오 버튼으로 막대·꺾은선·면적 차트를 선택해 평점 구간별 영화 수를 비교할 수 있다.

`movies.csv`는 별도로 준비된 파일을 사용한다. 앱과 같은 폴더에 두고, GitHub에 업로드할 때도 함께 올린다. 평점은 실제 관객 평가가 아닌 과제용 예시 값이며, 장르는 앱에서 비교하기 위한 대표 분류이다.


---

## STEP 2. 대시보드 만들기

아래 셀에 코드를 작성한다. 필수 조건 5가지를 모두 포함해야 한다.

막히면 아래 **힌트 코드**를 참고하자.

In [3]:
%%writefile streamlit_app.py
from pathlib import Path

import streamlit as st
import pandas as pd

st.set_page_config(page_title="오늘 뭐 볼까?", layout="centered")
st.title("오늘 뭐 볼까?")
st.caption("10강 실습 과제 · 평점은 실제 관객 평점이 아닌 과제용 예시 값입니다.")

# 앱과 같은 폴더에 있는 영화 목록을 읽습니다.
df = pd.read_csv(Path(__file__).with_name("movies.csv"))

# 힌트의 사이드바와 선택박스, 슬라이더를 활용합니다.
st.sidebar.title("영화 선택")
장르 = st.sidebar.selectbox("장르", ["전체"] + df["장르"].unique().tolist())
최소_평점 = st.sidebar.slider("최소 평점", 0.0, 10.0, 0.0, 0.5)
표시할_영화_수 = st.sidebar.slider("목록에 표시할 영화 수", 1, len(df), min(10, len(df)))

# 힌트의 라디오 버튼으로 차트 종류를 선택합니다.
차트_종류 = st.sidebar.radio("차트 종류", ["막대", "꺾은선", "면적"])

# 선택한 장르와 최소 평점을 모두 만족하는 영화만 남깁니다.
결과 = df[df["예시 평점"] >= 최소_평점]
if 장르 != "전체":
    결과 = 결과[결과["장르"] == 장르]
결과 = 결과.sort_values("예시 평점", ascending=False)

st.metric("조건에 맞는 영화 수", f"{len(결과)}편")

if 결과.empty:
    st.info("조건에 맞는 영화가 없습니다. 장르나 최소 평점을 바꿔 보세요.")
else:
    # 표시 개수는 영화 목록에만 적용합니다.
    표시할_영화 = 결과.head(표시할_영화_수)
    st.caption(f"조건에 맞는 {len(결과)}편 중 평점이 높은 {len(표시할_영화)}편을 보여줍니다.")
    st.header("영화 목록")
    st.dataframe(표시할_영화, hide_index=True, alt="선택 조건과 표시 개수에 맞는 영화 목록")

    st.header("평점 구간별 영화 수")
    st.caption(f"목록 표시 개수와 관계없이 선택 조건에 맞는 전체 {len(결과)}편을 집계합니다.")

    # 0점대는 0점 이상 1점 미만입니다. 마지막 구간은 10점도 포함합니다.
    평점_구간 = [f"{점수}점대" for 점수 in range(9)] + ["9~10점"]
    구간별_영화_수 = []
    for 하한 in range(10):
        if 하한 == 9:
            구간 = 결과[(결과["예시 평점"] >= 9) & (결과["예시 평점"] <= 10)]
        else:
            구간 = 결과[(결과["예시 평점"] >= 하한) & (결과["예시 평점"] < 하한 + 1)]
        구간별_영화_수.append(len(구간))

    차트_데이터 = pd.DataFrame({"평점 구간": 평점_구간, "영화 수": 구간별_영화_수}).set_index("평점 구간")
    if 차트_종류 == "막대":
        st.bar_chart(차트_데이터, alt="평점 구간별 영화 수 막대그래프")
    elif 차트_종류 == "꺾은선":
        st.line_chart(차트_데이터, alt="평점 구간별 영화 수 꺾은선그래프")
    else:
        st.area_chart(차트_데이터, alt="평점 구간별 영화 수 면적그래프")


Overwriting streamlit_app.py


---

### 힌트: 자주 쓰는 패턴 모음

위젯 값이 바뀌면 화면이 바뀌는 패턴은 이렇게 씁니다.

```python
# 슬라이더 값에 따라 차트 데이터 개수 변경
n = st.slider("데이터 개수", 5, 50, 20)
data = pd.DataFrame(np.random.randn(n, 2), columns=["A", "B"])
st.line_chart(data)

# 선택박스 값에 따라 다른 내용 표시
category = st.selectbox("카테고리", ["음식", "교통", "쇼핑"])
if category == "음식":
    st.write("음식 관련 내용")
elif category == "교통":
    st.write("교통 관련 내용")

# 라디오 버튼으로 차트 종류 선택
chart_type = st.radio("차트 종류", ["막대", "꺾은선", "면적"])
if chart_type == "막대":
    st.bar_chart(data)
elif chart_type == "꺾은선":
    st.line_chart(data)
else:
    st.area_chart(data)
```

사이드바에 위젯을 넣고 싶으면 이렇게 씁니다.

```python
st.sidebar.title("설정")
n = st.sidebar.slider("데이터 개수", 5, 50, 20)
```

---

## STEP 3. 로컬에서 실행 확인

터미널(메뉴 → Terminal → New Terminal)에서 아래 명령어를 실행한다.

```
streamlit run streamlit_app.py
```

**체크리스트**
- [x] 브라우저에서 앱이 열렸다
- [x] 텍스트, 데이터, 차트가 잘 보인다
- [x] 위젯을 조작하면 화면이 바뀐다
- [x] 필수 조건 5가지가 모두 포함되어 있다

---

## STEP 4. requirements.txt 생성

아래 셀을 실행한다. 추가 라이브러리를 사용했다면 해당 라이브러리도 추가한다.

> 예: `matplotlib`, `plotly`, `pydeck` 등을 사용했다면 아래에 한 줄씩 추가

In [4]:
%%writefile requirements.txt
streamlit>=1.66.1
pandas


Overwriting requirements.txt


**체크리스트**
- [x] `requirements.txt` 파일이 현재 폴더에 생성됐다
- [x] 내가 사용한 라이브러리가 모두 포함되어 있다

---

## STEP 5. GitHub 업로드

9강 과제에서 만든 저장소를 재사용하거나 새로 만든다.

**기존 저장소 재사용하는 경우:**
1. 기존 저장소 접속
2. `streamlit_app.py` 파일 클릭 → 연필 아이콘(Edit) 클릭 → 새 코드로 교체 → Commit
3. `requirements.txt`도 동일하게 업데이트

**새 저장소를 만드는 경우:**
1. GitHub에서 새 저장소 생성 (Public, README 추가)
2. `streamlit_app.py`와 `requirements.txt` 업로드
3. Commit

**체크리스트**
- [x] `streamlit_app.py`가 GitHub에 올라가 있다
- [x] `requirements.txt`가 GitHub에 올라가 있다

---

## STEP 6. Streamlit Cloud 배포

9강과 동일한 방식으로 배포한다.

1. https://share.streamlit.io 접속 후 로그인
2. **Create app** 클릭
3. GitHub 저장소, Branch, Main file path(`streamlit_app.py`) 선택
4. **Deploy** 클릭
5. 배포 완료 → URL 확인

> 기존 앱을 업데이트한 경우: GitHub에 코드를 올리면 Streamlit Cloud가 자동으로 재배포합니다. 별도 작업 불필요.

**체크리스트**
- [x] 배포가 완료되어 URL이 생성됐다
- [x] 그 URL로 접속하면 내 앱이 잘 보인다
- [x] 위젯을 조작해도 잘 동작한다

---

## 제출

아래 두 가지를 eTL 과제함에 제출한다.

1. **이 노트북 파일** (`10강_실습과제.ipynb`) — STEP 1의 "내 주제" 칸을 채워서 제출
2. **배포 URL** — 아래 칸에 입력

**🔗 내 앱 배포 URL**

https://jhbae003-my-streamlit-app-streamlit-app-ekqc9z.streamlit.app/


---

## 자주 생기는 문제

**Q. 배포 후 앱이 에러 화면으로 뜬다**

→ Streamlit Cloud의 로그를 확인하자. 보통 두 가지 원인이다.
- `requirements.txt`에 라이브러리가 누락된 경우 → 추가 후 GitHub에 다시 올리기
- 코드에 오류가 있는 경우 → 로컬에서 먼저 실행해서 확인하기

**Q. GitHub에 올렸는데 Streamlit Cloud가 자동으로 반영이 안 된다**

→ Streamlit Cloud 대시보드에서 앱 오른쪽 메뉴(⋮) → **Reboot app** 클릭

**Q. 위젯을 바꿔도 화면이 안 바뀐다**

→ 위젯 값을 변수에 저장하고, 그 변수를 화면 출력에 사용해야 한다.

```python
# 잘못된 예
st.slider("개수", 5, 50, 20)   # 변수에 저장 안 함
st.line_chart(data)             # 위젯 값이 반영 안 됨

# 올바른 예
n = st.slider("개수", 5, 50, 20)  # 변수에 저장
data = pd.DataFrame(np.random.randn(n, 2))
st.line_chart(data)                # n이 반영됨
```

**Q. 9강 과제 때 쓴 저장소에 올렸는데 이전 앱이 덮어써진다**

→ 정상이다. 같은 파일명(`streamlit_app.py`)으로 올리면 자동으로 교체된다. 9강 앱을 따로 보존하고 싶으면 새 저장소를 만들자.